# Piloto de coleta e medição — corpus de áudio

Executa a esteira completa do passo 4 do `docs/roadmap.md` em ambiente com GPU:
coleta, transcrição e diarização, seguidas das medições que justificam o piloto.

**O piloto não produz dados de pesquisa.** Ele produz os números que hoje são
suposições declaradas e que precisam virar medida antes de a coleta escalar:

1. **Rendimento por camada** — que fração da duração de um vídeo é fala, depois de
   descontar vinheta, música e silêncio. O planejamento assume 35% para vox-pop,
   60% para rádio e TV e 70% para vlog. Se o rendimento real do vox-pop for metade
   do suposto, a meta de 50 h dobra.
2. **Diarização** — quantos locutores distintos por arquivo e como o tempo se
   distribui entre eles. Na camada de vox-pop é o que separa o morador entrevistado
   do repórter, e disso depende a camada inteira.
3. **Dificuldade de transcrição por variedade** — se o reconhecimento erra mais em
   fala nordestina que em fala do Sudeste. Um WER honesto exige transcrição humana
   de referência; este notebook calcula um **indicador aproximado** e **exporta a
   amostra estratificada** para a transcrição manual. O indicador não substitui o
   WER: serve para dizer se vale a pena medi-lo a sério.

**Antes de rodar:** ative a GPU em *Ambiente de execução -> Alterar o tipo de ambiente*.

## 1. Verificação do ambiente

Executada primeiro, e de propósito. No teste local de 27/08/2026 duas condições
falharam de modo silencioso: a versão do `yt-dlp` estava desatualizada e não havia
runtime de JavaScript disponível. Nos dois casos a leitura de metadados seguia
funcionando e apenas o download falhava, de modo que o erro se disfarçava de
sucesso. Ver `docs/pendencias.md`, seção 4.6.

In [ ]:
import shutil, subprocess

def checar(nome, condicao, detalhe=""):
    print(f"{'OK   ' if condicao else 'FALHA'}  {nome}  {detalhe}")
    return condicao

ok = True
try:
    import torch
    ok &= checar("GPU", torch.cuda.is_available(),
                 torch.cuda.get_device_name(0) if torch.cuda.is_available()
                 else "sem GPU - ative em Ambiente de execucao")
except ImportError:
    print("torch ainda nao instalado; rode a proxima celula e volte aqui")

ok &= checar("ffmpeg", shutil.which("ffmpeg") is not None)

runtime = next((r for r in ("deno", "node", "bun") if shutil.which(r)), None)
ok &= checar("runtime de JavaScript", runtime is not None,
             runtime or "necessario para o download do YouTube")

try:
    v = subprocess.run(["yt-dlp", "--version"], capture_output=True, text=True).stdout.strip()
    ok &= checar("yt-dlp", v >= "2026.08.19", f"versao {v} (minimo 2026.08.19)")
except FileNotFoundError:
    print("FALHA  yt-dlp nao instalado")
    ok = False

print("\nAmbiente pronto." if ok else "\nCorrija os itens acima antes de prosseguir.")

## 2. Instalação

In [ ]:
!pip install -q "yt-dlp>=2026.08.19" "faster-whisper>=1.0.0" "pyannote.audio>=3.1.0" jiwer
!apt-get -qq install -y ffmpeg > /dev/null

# Runtime de JavaScript: o yt-dlp habilita apenas deno por padrao.
!curl -fsSL https://deno.land/install.sh | sh -s -- -y > /dev/null 2>&1
import os
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.deno/bin")
print("instalado; execute novamente a celula de verificacao")

## 3. Repositório e credenciais

O `HF_TOKEN` precisa ter aceitado os termos de
`pyannote/speaker-diarization-community-1`. Use o painel de segredos do Colab, o
ícone de chave na barra lateral, com o nome `HF_TOKEN`. Nunca cole o token numa
célula: ele ficaria gravado no notebook.

In [ ]:
!git clone -q https://github.com/Aryazinha/vies-nordeste-bertimbau.git
%cd vies-nordeste-bertimbau/pipeline_coleta_piloto

import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN carregado do painel de segredos")

## 4. Plano de coleta

`--piloto` reduz a meta a 15 minutos por camada e por estado. A regra de cobertura
mínima garante que todo canal contribua ao menos um trecho, de modo que o número de
falantes distintos não dependa da cota. Para um piloto ainda menor, use
`--max-canais 2`.

In [ ]:
!python selecionar_videos.py --piloto --max-canais 2 --saida plano_piloto.json

import json
plano = json.load(open("plano_piloto.json", encoding="utf-8"))
specs = plano["specs"]
horas = sum(s["duracao_coletada_s"] for s in specs) / 3600
canais = len({s["channel_id"] for s in specs})
print(f"\n{len(specs)} trechos, {horas:.2f} h, {canais} canais distintos")

## 5. Coleta

A verificação de existência do arquivo está dentro de `baixar_audio`. Divergência
entre coletados e planejados indica falha real, e não erro de contagem — foi
precisamente essa distinção que faltava antes da correção de 27/08/2026.

In [ ]:
from collect import coletar_lote

metas = coletar_lote(specs)
print(f"\n{len(metas)}/{len(specs)} coletados")

## 6. Transcrição e diarização

In [ ]:
import json
from config import AUDIO_DIR, FINAL_DIR
from transcribe import transcrever_audio
from diarize import diarizar_audio
from pipeline import _montar_registro_final

registros = []
for meta in metas:
    caminho = AUDIO_DIR / f"{meta.id}.wav"
    if not caminho.exists():
        print(f"ausente: {meta.id}")
        continue
    print(f"processando {meta.id} ({meta.estado_alvo}/{meta.tipo_fonte})")
    transcricao = transcrever_audio(caminho)
    turnos = diarizar_audio(caminho)
    registro = _montar_registro_final(meta, transcricao, turnos)
    (FINAL_DIR / f"{meta.id}.json").write_text(
        json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")
    registros.append(registro)

print(f"\n{len(registros)} registros completos")

## 7. Medições

### 7.1 Rendimento por camada

Compara a duração de fala detectada com a duração do arquivo. Substitui as frações
supostas no cálculo da meta de volume.

In [ ]:
from collections import defaultdict

rend = defaultdict(list)
for r in registros:
    segs = r["transcricao"]["segmentos"]
    if not segs:
        continue
    dur_arquivo = segs[-1]["end"]
    fala = sum(s["end"] - s["start"] for s in segs)
    if dur_arquivo:
        rend[r["tipo_fonte"]].append(fala / dur_arquivo)

SUPOSTO = {"entrevista_vox_pop": 0.35,
           "podcast_radio_tv_regional": 0.60,
           "vlog_amador": 0.70}

print(f"{'camada':34s} {'n':>3s} {'medido':>8s} {'suposto':>8s}")
for camada, vals in rend.items():
    print(f"{camada:34s} {len(vals):3d} {sum(vals)/len(vals):8.1%} {SUPOSTO.get(camada, 0):8.1%}")

print("\nEsta medida desconta silencio e musica, mas nao a fala de locutor de outra")
print("variedade. O desconto do reporter aparece na secao 7.2.")

### 7.2 Locutores por arquivo

Na camada de vox-pop o pressuposto é que a diarização separe o morador entrevistado
do repórter. Um arquivo dessa camada com um único locutor detectado indica falha da
diarização ou vídeo sem entrevista — e, nos dois casos, material que não serve.

In [ ]:
for r in registros:
    tempos = defaultdict(float)
    for t in r["diarizacao"]:
        tempos[t["speaker"]] += t["end"] - t["start"]
    total = sum(tempos.values()) or 1
    dist = ", ".join(f"{s}={d/total:.0%}"
                     for s, d in sorted(tempos.items(), key=lambda x: -x[1]))
    print(f"{r['estado_alvo']}/{r['tipo_fonte'][:12]:12s} {r['id']}  "
          f"{len(tempos)} locutor(es)  [{dist}]")

### 7.3 Indicador aproximado de dificuldade de transcrição

Confiança média por palavra, agregada por estado. **Não é WER.** É um indicador
fraco, que mede a certeza do modelo e não o acerto. Serve para apontar se existe
diferença sistemática entre variedades que justifique a transcrição manual de
referência. Diferença consistente entre o grupo nordestino e o de controle é
resultado a investigar, jamais a reportar como WER.

In [ ]:
conf = defaultdict(list)
for r in registros:
    for seg in r["transcricao"]["segmentos"]:
        for w in seg["words"]:
            conf[r["estado_alvo"]].append(w["probability"])

print(f"{'estado':8s} {'palavras':>9s} {'confianca media':>17s}")
for uf in ["PB", "PE", "CE", "BA", "SP", "RJ"]:
    if conf[uf]:
        print(f"{uf:8s} {len(conf[uf]):9d} {sum(conf[uf])/len(conf[uf]):17.3f}")

ne = [p for uf in ("PB", "PE", "CE", "BA") for p in conf[uf]]
se = [p for uf in ("SP", "RJ") for p in conf[uf]]
if ne and se:
    mne, mse = sum(ne)/len(ne), sum(se)/len(se)
    print(f"\nNordeste {mne:.3f}  |  Sudeste {mse:.3f}  |  diferenca {mne-mse:+.3f}")

### 7.4 Amostra para transcrição manual

O WER só pode ser calculado contra transcrição humana. Esta célula exporta trechos
totalizando 20 minutos por estado, estratificados entre camadas, conforme
`docs/pendencias.md`. Transcreva à mão o campo `referencia_manual` e compare com
`jiwer`.

In [ ]:
import random
random.seed(20260827)

por_estado = defaultdict(list)
for r in registros:
    for seg in r["transcricao"]["segmentos"]:
        if seg["end"] - seg["start"] >= 5:
            por_estado[r["estado_alvo"]].append({
                "id": r["id"],
                "estado": r["estado_alvo"],
                "camada": r["tipo_fonte"],
                "inicio_s": round(seg["start"], 2),
                "fim_s": round(seg["end"], 2),
                "hipotese_asr": seg["text"].strip(),
                "referencia_manual": "",
            })

selecao = []
for uf, itens in por_estado.items():
    random.shuffle(itens)
    acumulado, escolhidos = 0.0, []
    for i in itens:
        if acumulado >= 20 * 60:
            break
        escolhidos.append(i)
        acumulado += i["fim_s"] - i["inicio_s"]
    selecao.extend(escolhidos)
    print(f"{uf}: {len(escolhidos)} trechos, {acumulado/60:.1f} min")

with open("amostra_wer.json", "w", encoding="utf-8") as f:
    json.dump(selecao, f, ensure_ascii=False, indent=2)
print(f"\n{len(selecao)} trechos em amostra_wer.json - preencha referencia_manual")

## 8. Exportação

**Não baixe o áudio.** A seção 1.4.2 do `CLAUDE.md` estabelece que o áudio bruto não
é redistribuído, e a coleta completa chega a cerca de 6 GB. O que precisa voltar são
as transcrições e os números.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("piloto_resultados", "zip", FINAL_DIR)
files.download("piloto_resultados.zip")
files.download("amostra_wer.json")